# CallSense — Member 3 | 3. Arabic Text-to-Speech
هذه النوت بوك تحمّل ملخص E3 أو نصًا تجريبيًا وتحوله إلى صوت عربي باستخدام نموذج Hugging Face `facebook/mms-tts-ara`.

يفضّل T4 GPU. النموذج يحتاج تنزيلًا أول مرة من Hugging Face، لذلك يلزم اتصال إنترنت أثناء التحميل.

## 1) تثبيت المكتبات

In [1]:
!pip -q install transformers scipy soundfile

## 2) تحميل نموذج TTS العربي

In [2]:
import torch, numpy as np, re, soundfile as sf
from transformers import VitsModel, AutoTokenizer
from IPython.display import Audio, display

MODEL_NAME = "facebook/mms-tts-ara"
device = "cuda" if torch.cuda.is_available() else "cpu"
tts_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tts_model = VitsModel.from_pretrained(MODEL_NAME).to(device)
tts_model.eval()
sample_rate = int(tts_model.config.sampling_rate)
print("Device:", device, "| Sample rate:", sample_rate)

config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  145MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Device: cuda | Sample rate: 16000


## 3) إدخال الملخص

In [3]:
# الخيار الأول: اكتبي نصًا هنا
digest_text = """
ملخص تجريبي: وردت استفسارات من العملاء حول الفواتير والخدمات.
يرجى مراجعة التسجيلات المشار إليها والتحقق من تفاصيل كل حالة.
"""

# الخيار الثاني: ارفعي digest.txt من النوت بوك السابقة ثم شغّلي:
from google.colab import files
uploaded = files.upload()
name = next(iter(uploaded))
digest_text = open(name, encoding="utf-8").read()

print(digest_text)

Saving digest (1).txt to digest (1).txt
ملخص الشكاوى الواردة من العملاء:

1. [S1] و[S2] يشكون عدم وجود إمكانية لإلغاء رسوم التجوال أو الإنترنت التي لم يقوموا بطلبها، كما يشكون أيضاً من تأخير الردود على الشكاوى [S1] و[S2].

2. [S3] يشكو عدم حل مشكلة ضعف خدمة الإنترنت رغم تقديم بلاغات عدة، مما أدى إلى استمرار رسوم الخدمة [S3].

3. [S4] يشكو من تكرار الرسائل التسويقية وعدم استجابتها، بالإضافة إلى تأخير الردود على الشكاوى [S4].

4. [S5] يشكو من تأخير إرسال فاتورة الشهر [S5].

5. [S6] يشكو من الرسائل التسويقية المستمرة رغم إيقافها [S6].

6. [S7] يشكو من عدم استجابة الشركة لطلبات الغاء الاشتراك، مما أدى إلى دفع رسوم بدون خدمات [S7].

7. [S8] يشكو من تأخير إرسال فاتورة الشهر، مما أدى إلى عدم قدرته على دفعها [S8].


## 4) تنظيف النص وتقسيمه لجمل قصيرة

In [4]:
def clean_for_speech(text):
    text = re.sub(r"\\[S\\d+\\]", "", str(text))  # إزالة رموز citations
    text = re.sub(r"(?m)^\\s*[-•]\\s*", "", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

def split_sentences(text, max_chars=220):
    text = clean_for_speech(text)
    pieces = re.split(r"(?<=[.!؟?])\\s+", text)
    result, current = [], ""
    for p in pieces:
        if not p:
            continue
        if len(current) + len(p) + 1 <= max_chars:
            current = (current + " " + p).strip()
        else:
            if current: result.append(current)
            current = p
    if current: result.append(current)
    return result

sentences = split_sentences(digest_text)
sentences

['ملخص الشكاوى الواردة من العملاء:\n\n1. [S1] و[S2] يشكون عدم وجود إمكانية لإلغاء رسوم التجوال أو الإنترنت التي لم يقوموا بطلبها، كما يشكون أيضاً من تأخير الردود على الشكاوى [S1] و[S2].\n\n2. [S3] يشكو عدم حل مشكلة ضعف خدمة الإنترنت رغم تقديم بلاغات عدة، مما أدى إلى استمرار رسوم الخدمة [S3].\n\n3. [S4] يشكو من تكرار الرسائل التسويقية وعدم استجابتها، بالإضافة إلى تأخير الردود على الشكاوى [S4].\n\n4. [S5] يشكو من تأخير إرسال فاتورة الشهر [S5].\n\n5. [S6] يشكو من الرسائل التسويقية المستمرة رغم إيقافها [S6].\n\n6. [S7] يشكو من عدم استجابة الشركة لطلبات الغاء الاشتراك، مما أدى إلى دفع رسوم بدون خدمات [S7].\n\n7. [S8] يشكو من تأخير إرسال فاتورة الشهر، مما أدى إلى عدم قدرته على دفعها [S8].']

## 5) توليد الصوت

In [5]:
def synthesize_arabic(text, output_path="digest.wav"):
    sentences = split_sentences(text)
    audio_parts = []
    for sentence in sentences:
        inputs = tts_tokenizer(sentence, return_tensors="pt")
        inputs = {k:v.to(device) for k,v in inputs.items()}
        with torch.no_grad():
            waveform = tts_model(**inputs).waveform.squeeze().cpu().numpy()
        audio_parts.append(waveform.astype(np.float32))
        # فاصل صمت قصير بين الجمل
        audio_parts.append(np.zeros(int(sample_rate * 0.25), dtype=np.float32))
    if not audio_parts:
        raise ValueError("النص فارغ بعد التنظيف.")
    audio = np.concatenate(audio_parts)
    sf.write(output_path, audio, sample_rate)
    return output_path, audio

wav_path, audio = synthesize_arabic(digest_text)
print("Saved:", wav_path, "| Duration:", round(len(audio)/sample_rate, 2), "seconds")
display(Audio(audio, rate=sample_rate))

Saved: digest.wav | Duration: 66.79 seconds


## 6) تنزيل الصوت

In [ ]:
from google.colab import files
files.download("digest.wav")

## ملاحظات التقييم
استمعي للصوت وقيّمي وضوح النطق، سلامة الكلمات العربية، الأسماء والأرقام، وسرعة القراءة. سجّلي زمن التوليد ومدة الصوت إذا كنتِ تقيسين latency أو RTF. لا تفترضي أن نطق الأرقام والكلمات الإنجليزية سيكون صحيحًا دائمًا.